In [ ]:
import os
# ============================================================
# ABLATION STUDY
# Post-training score-level ablation
# Seed 42
#
# Paper subsections:
# 1. Contribution of Representation and Detector Components
# 2. Contribution of Spatial, Contextual, and Reliability Modules
# ============================================================

from pathlib import Path
from collections import OrderedDict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from openpyxl import load_workbook
from openpyxl.styles import Font, Alignment
from openpyxl.utils import get_column_letter


# ============================================================
# 1. Experimental configuration
# ============================================================

SEED = 42

ROOT_DIR = Path(
    os.environ.get("GIS_ANOMALY_WORKSPACE_ROOT", ".")
    r"\Outlier Paper\Code"
)

CLEAN_RESULTS_DIR = (
    ROOT_DIR
    / "03_Proposed_Results_Seed_42"
)

COMPARISON_RESULTS_DIR = (
    ROOT_DIR
    / "04_Baseline_Comparison_Results_Seed_42"
)

INJECTED_RESULTS_DIR = (
    ROOT_DIR
    / "03_Injected_Proposed_Results_Seed_42"
    / "Injected_Datasets_with_0.1%_Injection_Rate"
)

OUTPUT_DIR = (
    ROOT_DIR
    / "06_Ablation_Study_Results_Seed_42"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# Internal anomaly flags used in reliability estimation
INTERNAL_SIGNAL_PERCENTILE = 98.0

# Exact final evaluation budgets
BUDGETS = OrderedDict(
    [
        ("0.10%", 0.0010),
        ("0.25%", 0.0025),
        ("0.50%", 0.0050),
        ("1.00%", 0.0100),
        ("2.00%", 0.0200),
    ]
)

SCENARIOS = OrderedDict(
    [
        ("extreme", "Extreme"),
        ("noise_strong", "Strong noise"),
        ("noise_moderate", "Moderate noise"),
        ("noise_mild", "Mild noise"),
        ("inconsistent", "Cross-feature inconsistency"),
    ]
)


# ============================================================
# 2. Component score files
# ============================================================

COMPONENT_FILES = OrderedDict(
    [
        (
            "ae",
            "ae_score_normalized.npy",
        ),
        (
            "if_feature",
            "if_feature_score_normalized.npy",
        ),
        (
            "lof_feature",
            "lof_feature_score_normalized.npy",
        ),
        (
            "md_feature",
            "md_feature_score_normalized.npy",
        ),
        (
            "if_embedding",
            "if_embedding_score_normalized.npy",
        ),
        (
            "lof_embedding",
            "lof_embedding_score_normalized.npy",
        ),
        (
            "md_embedding",
            "md_embedding_score_normalized.npy",
        ),
        (
            "spatial",
            "spatial_score_normalized.npy",
        ),
        (
            "pure_spatial",
            "pure_spatial_score_normalized.npy",
        ),
        (
            "context",
            "context_score_normalized.npy",
        ),
    ]
)


FEATURE_DETECTOR_KEYS = [
    "if_feature",
    "lof_feature",
    "md_feature",
]

EMBEDDING_DETECTOR_KEYS = [
    "if_embedding",
    "lof_embedding",
    "md_embedding",
]

CLASSICAL_DETECTOR_KEYS = (
    FEATURE_DETECTOR_KEYS
    + EMBEDDING_DETECTOR_KEYS
)

FULL_BASE_KEYS = (
    ["ae"]
    + CLASSICAL_DETECTOR_KEYS
)

IF_FAMILY_KEYS = [
    "if_feature",
    "if_embedding",
]

LOF_FAMILY_KEYS = [
    "lof_feature",
    "lof_embedding",
]

MD_FAMILY_KEYS = [
    "md_feature",
    "md_embedding",
]


# ============================================================
# 3. Required reference files
# ============================================================

INJECTED_INDICES_PATH = (
    INJECTED_RESULTS_DIR
    / "inject_indices.npy"
)

FULL_CLEAN_PRECLIP_PATH = (
    COMPARISON_RESULTS_DIR
    / "Full_final_fused_score_preclip.npy"
)

PURE_CLEAN_PRECLIP_PATH = (
    COMPARISON_RESULTS_DIR
    / "Pure_final_fused_score_preclip.npy"
)

FULL_SPATIAL_PARAMS_PATH = (
    CLEAN_RESULTS_DIR
    / "full_spatial_normalization_params.npz"
)

FULL_PURE_PARAMS_PATH = (
    CLEAN_RESULTS_DIR
    / "full_pure_final_normalization_params.npz"
)

CLEAN_RELIABILITY_PATH = (
    CLEAN_RESULTS_DIR
    / "full_pure_reliability_diagnostics.npz"
)

CLEAN_DETECTOR_LABELS_PATH = (
    CLEAN_RESULTS_DIR
    / "detector_anomaly_labels.npz"
)


# ============================================================
# 4. Basic file and array validation
# ============================================================

def require_file(path, description):
    """Raise a clear error when a required file is absent."""

    if not path.exists():
        raise FileNotFoundError(
            f"Missing {description}:\n"
            f"{path.resolve()}"
        )


required_global_files = [
    (
        INJECTED_INDICES_PATH,
        "injected-index file",
    ),
    (
        FULL_CLEAN_PRECLIP_PATH,
        "clean Full pre-clipping score",
    ),
    (
        PURE_CLEAN_PRECLIP_PATH,
        "clean Pure pre-clipping score",
    ),
    (
        FULL_SPATIAL_PARAMS_PATH,
        "Full spatial normalization parameters",
    ),
    (
        FULL_PURE_PARAMS_PATH,
        "Full/Pure normalization parameters",
    ),
    (
        CLEAN_RELIABILITY_PATH,
        "clean reliability diagnostics",
    ),
    (
        CLEAN_DETECTOR_LABELS_PATH,
        "clean detector-label archive",
    ),
]

for file_path, file_description in required_global_files:
    require_file(
        file_path,
        file_description,
    )


for component_name, file_name in COMPONENT_FILES.items():
    require_file(
        CLEAN_RESULTS_DIR / file_name,
        f"clean {component_name} score",
    )


for scenario_folder in SCENARIOS:

    current_scenario_dir = (
        INJECTED_RESULTS_DIR
        / scenario_folder
    )

    if not current_scenario_dir.exists():
        raise FileNotFoundError(
            "Missing injected scenario folder:\n"
            f"{current_scenario_dir.resolve()}"
        )

    for component_name, file_name in COMPONENT_FILES.items():
        require_file(
            current_scenario_dir / file_name,
            (
                f"{scenario_folder} "
                f"{component_name} score"
            ),
        )

    require_file(
        current_scenario_dir
        / "detector_anomaly_labels.npz",
        (
            f"{scenario_folder} detector-label archive"
        ),
    )

    require_file(
        current_scenario_dir
        / "full_final_anomaly_score_raw.npy",
        (
            f"{scenario_folder} Full raw final score"
        ),
    )

    require_file(
        current_scenario_dir
        / "pure_final_anomaly_score_raw.npy",
        (
            f"{scenario_folder} Pure raw final score"
        ),
    )


print("All required ablation files are available.")


# ============================================================
# 5. Loading utilities
# ============================================================

def load_vector(
    path,
    expected_length=None,
    name="array",
    mmap=True,
):
    """Load and validate a finite one-dimensional array."""

    array = np.load(
        path,
        mmap_mode="r" if mmap else None,
    )

    array = np.asarray(
        array
    ).reshape(-1)

    if expected_length is not None:
        if array.size != expected_length:
            raise ValueError(
                f"{name} has length {array.size:,}; "
                f"expected {expected_length:,}."
            )

    if not np.isfinite(array).all():
        raise ValueError(
            f"{name} contains NaN or infinite values."
        )

    return array


def load_component_scores(
    folder,
    expected_length=None,
):
    """Load normalized component scores, including separate Full and Pure spatial scores."""

    loaded_scores = OrderedDict()

    for component_name, file_name in COMPONENT_FILES.items():

        loaded_scores[component_name] = load_vector(
            folder / file_name,
            expected_length=expected_length,
            name=f"{folder.name}: {component_name}",
            mmap=True,
        )

    lengths = {
        score.size
        for score in loaded_scores.values()
    }

    if len(lengths) != 1:
        raise ValueError(
            f"Component lengths differ in {folder.resolve()}."
        )

    return loaded_scores


# ============================================================
# 6. Load clean components and experiment size
# ============================================================

clean_scores = load_component_scores(
    CLEAN_RESULTS_DIR
)

N_SAMPLES = clean_scores["ae"].size

inject_indices = load_vector(
    INJECTED_INDICES_PATH,
    name="injected indices",
    mmap=False,
).astype(
    np.int64,
    copy=False,
)

if inject_indices.size == 0:
    raise ValueError(
        "The injected-index array is empty."
    )

if np.unique(inject_indices).size != inject_indices.size:
    raise ValueError(
        "The injected-index array contains duplicates."
    )

if (
    inject_indices.min() < 0
    or inject_indices.max() >= N_SAMPLES
):
    raise ValueError(
        "The injected-index array contains "
        "out-of-range indices."
    )

injected_mask = np.zeros(
    N_SAMPLES,
    dtype=bool,
)

injected_mask[
    inject_indices
] = True

N_INJECTED = int(
    inject_indices.size
)

ACTUAL_INJECTION_RATIO = (
    N_INJECTED
    / N_SAMPLES
)

print(f"Number of samples: {N_SAMPLES:,}")
print(f"Injected samples: {N_INJECTED:,}")
print(
    "Actual injection ratio: "
    f"{ACTUAL_INJECTION_RATIO * 100:.6f}%"
)


# ============================================================
# 7. Load the frozen clean fusion parameters
# ============================================================

with np.load(
    FULL_SPATIAL_PARAMS_PATH
) as full_spatial_parameters:

    FULL_SPATIAL_P1 = float(
        full_spatial_parameters["p1"]
    )

    FULL_SPATIAL_P99 = float(
        full_spatial_parameters["p99"]
    )

    FULL_BASE_WEIGHT = float(
        full_spatial_parameters["base_weight"]
    )

    FULL_SPATIAL_WEIGHT = float(
        full_spatial_parameters["spatial_weight"]
    )


with np.load(
    FULL_PURE_PARAMS_PATH
) as full_pure_parameters:

    PURE_SPATIAL_P1 = float(
        full_pure_parameters["pure_spatial_p1"]
    )

    PURE_SPATIAL_P99 = float(
        full_pure_parameters["pure_spatial_p99"]
    )

    PURE_BASE_WEIGHT = float(
        full_pure_parameters["pure_base_weight"]
    )

    PURE_SPATIAL_WEIGHT = float(
        full_pure_parameters["pure_spatial_weight"]
    )

    AGREEMENT_WEIGHT = float(
        full_pure_parameters["consensus_weight"]
    )

    CERTAINTY_WEIGHT = float(
        full_pure_parameters["certainty_weight"]
    )


if FULL_SPATIAL_P99 <= FULL_SPATIAL_P1:
    raise ValueError(
        "Invalid clean Full spatial bounds."
    )

if PURE_SPATIAL_P99 <= PURE_SPATIAL_P1:
    raise ValueError(
        "Invalid clean Pure spatial bounds."
    )

if not np.isclose(
    AGREEMENT_WEIGHT + CERTAINTY_WEIGHT,
    1.0,
    atol=1e-6,
):
    raise ValueError(
        "Reliability weights do not sum to one."
    )


print("\nFrozen model parameters:")
print(
    "Full fusion weights:",
    FULL_BASE_WEIGHT,
    FULL_SPATIAL_WEIGHT,
)
print(
    "Pure fusion weights:",
    PURE_BASE_WEIGHT,
    PURE_SPATIAL_WEIGHT,
)
print(
    "Reliability weights:",
    AGREEMENT_WEIGHT,
    CERTAINTY_WEIGHT,
)


# ============================================================
# 8. Load anomaly flags used by the reliability mechanism
# ============================================================

DETECTOR_ARCHIVE_KEYS = {
    "ae": "ae",
    "if_feature": "if_feature",
    "lof_feature": "lof_feature",
    "md_feature": "md_feature",
    "if_embedding": "if_embedding",
    "lof_embedding": "lof_embedding",
    "md_embedding": "md_embedding",
}


def exact_top_fraction_labels(
    score,
    fraction=0.02,
):
    """
    Construct deterministic exact top-fraction binary labels.

    Larger scores indicate stronger anomalousness. Exact ties at the
    selection boundary are resolved by ascending pixel index so that
    exactly ceil(fraction * N) observations are selected.
    """

    score = np.asarray(score).reshape(-1)

    if not np.isfinite(score).all():
        raise ValueError(
            "Score contains NaN or infinite values."
        )

    if not 0.0 < fraction < 1.0:
        raise ValueError(
            "fraction must lie between 0 and 1."
        )

    n_select = int(
        np.ceil(
            fraction * score.size
        )
    )

    partition_position = (
        score.size
        - n_select
    )

    threshold = np.partition(
        score,
        partition_position,
    )[partition_position]

    above_threshold = np.flatnonzero(
        score > threshold
    )

    tied_at_threshold = np.flatnonzero(
        score == threshold
    )

    number_of_ties_needed = (
        n_select
        - above_threshold.size
    )

    selected_indices = np.concatenate(
        [
            above_threshold,
            tied_at_threshold[
                :number_of_ties_needed
            ],
        ]
    )

    if selected_indices.size != n_select:
        raise RuntimeError(
            "Exact top-fraction labeling returned an "
            "incorrect number of observations."
        )

    labels = np.zeros(
        score.size,
        dtype=np.uint8,
    )

    labels[
        selected_indices
    ] = 1

    return labels, float(threshold)


def load_internal_signal_labels(
    folder,
    scores,
    expected_length,
):
    """
    Load saved detector labels and construct the spatial and
    contextual labels using the original percentile rule.
    """

    label_archive_path = (
        folder
        / "detector_anomaly_labels.npz"
    )

    require_file(
        label_archive_path,
        "detector anomaly-label archive",
    )

    internal_labels = OrderedDict()
    threshold_rows = []

    with np.load(
        label_archive_path
    ) as archive:

        available_keys = set(
            archive.files
        )

        for score_key, archive_key in (
            DETECTOR_ARCHIVE_KEYS.items()
        ):

            if archive_key not in available_keys:
                raise KeyError(
                    f"Missing key '{archive_key}' in:\n"
                    f"{label_archive_path.resolve()}\n"
                    f"Available keys: {sorted(available_keys)}"
                )

            label_array = np.asarray(
                archive[archive_key]
            ).reshape(-1)

            if label_array.size != expected_length:
                raise ValueError(
                    f"{score_key} labels have invalid length."
                )

            unique_values = np.unique(
                label_array
            )

            if not set(
                unique_values.tolist()
            ).issubset({0, 1}):
                raise ValueError(
                    f"{score_key} labels are not binary."
                )

            internal_labels[score_key] = (
                label_array.astype(
                    np.uint8,
                    copy=False,
                )
            )

            threshold_rows.append(
                {
                    "Signal": score_key,
                    "Threshold source":
                        "Saved detector label archive",
                    "Threshold": np.nan,
                    "Positive labels": int(
                        label_array.sum()
                    ),
                    "Positive share (%)": float(
                        label_array.mean()
                        * 100.0
                    ),
                }
            )

    for score_key in [
        "spatial",
        "pure_spatial",
        "context",
    ]:

        labels, threshold = (
            exact_top_fraction_labels(
                scores[score_key],
                fraction=0.02,
            )
        )

        internal_labels[score_key] = labels

        threshold_rows.append(
            {
                "Signal": score_key,
                "Threshold source":
                    "Deterministic exact top-2% selection",
                "Threshold": threshold,
                "Positive labels": int(
                    labels.sum()
                ),
                "Positive share (%)": float(
                    labels.mean()
                    * 100.0
                ),
            }
        )

    return (
        internal_labels,
        pd.DataFrame(
            threshold_rows
        ),
    )


clean_labels, clean_threshold_df = (
    load_internal_signal_labels(
        CLEAN_RESULTS_DIR,
        clean_scores,
        N_SAMPLES,
    )
)


# ============================================================
# 9. Score-construction utilities
# ============================================================

def average_scores(
    score_dictionary,
    score_keys,
):
    """Compute an equal-weight mean without stacking all arrays."""

    if len(score_keys) == 0:
        raise ValueError(
            "At least one score is required."
        )

    result = np.zeros(
        N_SAMPLES,
        dtype=np.float32,
    )

    contribution_weight = np.float32(
        1.0 / len(score_keys)
    )

    for score_key in score_keys:

        result += (
            np.asarray(
                score_dictionary[score_key],
                dtype=np.float32,
            )
            * contribution_weight
        )

    return result


def normalize_using_bounds(
    score,
    lower_bound,
    upper_bound,
):
    """Clip and normalize using clean-data bounds."""

    if upper_bound <= lower_bound:
        raise ValueError(
            "Upper normalization bound must exceed "
            "the lower bound."
        )

    normalized = (
        np.clip(
            score,
            lower_bound,
            upper_bound,
        )
        - lower_bound
    ) / (
        upper_bound
        - lower_bound
    )

    return normalized.astype(
        np.float32,
        copy=False,
    )


def calculate_uncertainty(
    score_dictionary,
    score_keys,
):
    """
    Calculate per-pixel standard deviation without constructing
    a large two-dimensional score stack.
    """

    if len(score_keys) == 0:
        raise ValueError(
            "Uncertainty requires at least one score."
        )

    score_sum = np.zeros(
        N_SAMPLES,
        dtype=np.float32,
    )

    squared_score_sum = np.zeros(
        N_SAMPLES,
        dtype=np.float32,
    )

    for score_key in score_keys:

        current_score = np.asarray(
            score_dictionary[score_key],
            dtype=np.float32,
        )

        score_sum += current_score
        squared_score_sum += (
            current_score
            * current_score
        )

    number_of_signals = np.float32(
        len(score_keys)
    )

    score_mean = (
        score_sum
        / number_of_signals
    )

    score_variance = (
        squared_score_sum
        / number_of_signals
        - score_mean
        * score_mean
    )

    score_variance = np.maximum(
        score_variance,
        0.0,
    )

    uncertainty = np.sqrt(
        score_variance
    )

    return uncertainty.astype(
        np.float32,
        copy=False,
    )


def construct_reliability(
    score_dictionary,
    label_dictionary,
    reliability_keys,
    uncertainty_bounds=None,
):
    """
    Construct reliability from anomaly agreement and certainty.

    When uncertainty_bounds is None, clean-data bounds are
    estimated. For injected data, the clean bounds are supplied.
    """

    if len(reliability_keys) == 0:
        raise ValueError(
            "Reliability requires at least one signal."
        )

    agreement_count = np.zeros(
        N_SAMPLES,
        dtype=np.uint16,
    )

    for score_key in reliability_keys:

        agreement_count += (
            label_dictionary[score_key]
            .astype(
                np.uint16,
                copy=False,
            )
        )

    normalized_agreement = (
        agreement_count.astype(
            np.float32,
            copy=False,
        )
        / np.float32(
            len(reliability_keys)
        )
    )

    uncertainty_raw = calculate_uncertainty(
        score_dictionary,
        reliability_keys,
    )

    if uncertainty_bounds is None:

        uncertainty_min = float(
            uncertainty_raw.min()
        )

        uncertainty_max = float(
            uncertainty_raw.max()
        )

    else:

        uncertainty_min = float(
            uncertainty_bounds[0]
        )

        uncertainty_max = float(
            uncertainty_bounds[1]
        )

    if uncertainty_max <= uncertainty_min:
        raise ValueError(
            "Invalid uncertainty normalization bounds."
        )

    uncertainty_normalized = (
        normalize_using_bounds(
            uncertainty_raw,
            uncertainty_min,
            uncertainty_max,
        )
    )

    reliability = (
        AGREEMENT_WEIGHT
        * normalized_agreement
        + CERTAINTY_WEIGHT
        * (
            1.0
            - uncertainty_normalized
        )
    )

    reliability = np.clip(
        reliability,
        0.0,
        1.0,
    ).astype(
        np.float32,
        copy=False,
    )

    return (
        reliability,
        (
            uncertainty_min,
            uncertainty_max,
        ),
    )


def construct_variant_score(
    score_dictionary,
    label_dictionary,
    specification,
    calibration=None,
):
    """
    Construct one score-level ablation variant.

    calibration=None:
        derive calibration from the clean dataset.

    calibration=dict:
        apply the previously derived clean calibration to
        an injected dataset.
    """

    base_keys = list(
        specification["base_keys"]
    )

    base_score = average_scores(
        score_dictionary,
        base_keys,
    )

    # Full and Pure use different spatial evidence.
    # Full spatial is derived from the seven-signal Full base, whereas
    # Pure spatial is derived only from the MVAE reconstruction score.
    spatial_key = (
        "pure_spatial"
        if specification.get("family") == "pure"
        else "spatial"
    )

    spatial_bounds = None

    if specification["use_spatial"]:

        spatial_fusion_raw = (
            specification["base_weight"]
            * base_score
            + specification["spatial_weight"]
            * np.asarray(
                score_dictionary[spatial_key],
                dtype=np.float32,
            )
        ).astype(
            np.float32,
            copy=False,
        )

        fixed_spatial_bounds = (
            specification.get(
                "fixed_spatial_bounds"
            )
        )

        if fixed_spatial_bounds is not None:

            spatial_bounds = (
                float(
                    fixed_spatial_bounds[0]
                ),
                float(
                    fixed_spatial_bounds[1]
                ),
            )

        elif calibration is not None:

            spatial_bounds = (
                float(
                    calibration["spatial_p1"]
                ),
                float(
                    calibration["spatial_p99"]
                ),
            )

        else:

            spatial_bounds = tuple(
                float(value)
                for value in np.percentile(
                    spatial_fusion_raw,
                    [
                        1,
                        99,
                    ],
                )
            )

        pre_reliability_score = (
            normalize_using_bounds(
                spatial_fusion_raw,
                spatial_bounds[0],
                spatial_bounds[1],
            )
        )

    else:

        pre_reliability_score = (
            base_score.astype(
                np.float32,
                copy=False,
            )
        )

    uncertainty_bounds = None
    reliability_signal_count = 0

    if specification["use_reliability"]:

        reliability_keys = list(
            base_keys
        )

        if specification["use_spatial"]:
            reliability_keys.append(
                spatial_key
            )

        if specification["use_context"]:
            reliability_keys.append(
                "context"
            )

        reliability_signal_count = len(
            reliability_keys
        )

        supplied_uncertainty_bounds = None

        if calibration is not None:

            supplied_uncertainty_bounds = (
                float(
                    calibration[
                        "uncertainty_min"
                    ]
                ),
                float(
                    calibration[
                        "uncertainty_max"
                    ]
                ),
            )

        reliability, uncertainty_bounds = (
            construct_reliability(
                score_dictionary,
                label_dictionary,
                reliability_keys,
                uncertainty_bounds=(
                    supplied_uncertainty_bounds
                ),
            )
        )

        final_score = (
            pre_reliability_score
            * reliability
        ).astype(
            np.float32,
            copy=False,
        )

    else:

        final_score = (
            pre_reliability_score.astype(
                np.float32,
                copy=False,
            )
        )

    if not np.isfinite(
        final_score
    ).all():
        raise ValueError(
            f"Non-finite score generated for "
            f"{specification['description']}."
        )

    derived_calibration = {
        "spatial_p1": (
            spatial_bounds[0]
            if spatial_bounds is not None
            else np.nan
        ),
        "spatial_p99": (
            spatial_bounds[1]
            if spatial_bounds is not None
            else np.nan
        ),
        "uncertainty_min": (
            uncertainty_bounds[0]
            if uncertainty_bounds is not None
            else np.nan
        ),
        "uncertainty_max": (
            uncertainty_bounds[1]
            if uncertainty_bounds is not None
            else np.nan
        ),
        "base_signal_count": len(
            base_keys
        ),
        "reliability_signal_count": (
            reliability_signal_count
        ),
    }

    return (
        final_score,
        derived_calibration,
    )


# ============================================================
# 10. Deterministic exact top-k ranking
# ============================================================

def exact_top_k_order(
    score,
    number_to_select,
):
    """
    Return exactly k indices ordered by:
    1. decreasing anomaly score;
    2. increasing pixel index for tied scores.
    """

    score = np.asarray(
        score
    ).reshape(-1)

    number_of_samples = (
        score.size
    )

    if not (
        1
        <= number_to_select
        <= number_of_samples
    ):
        raise ValueError(
            "Invalid exact top-k selection size."
        )

    partition_position = (
        number_of_samples
        - number_to_select
    )

    threshold = np.partition(
        score,
        partition_position,
    )[partition_position]

    above_threshold = np.flatnonzero(
        score > threshold
    )

    tied_at_threshold = np.flatnonzero(
        score == threshold
    )

    number_of_ties_needed = (
        number_to_select
        - above_threshold.size
    )

    if number_of_ties_needed < 0:
        raise RuntimeError(
            "Unexpected exact top-k partition result."
        )

    selected_indices = np.concatenate(
        [
            above_threshold,
            tied_at_threshold[
                :number_of_ties_needed
            ],
        ]
    )

    if selected_indices.size != number_to_select:
        raise RuntimeError(
            "Exact top-k selection returned "
            "an incorrect number of indices."
        )

    ranking_order = np.lexsort(
        (
            selected_indices,
            -score[selected_indices],
        )
    )

    return selected_indices[
        ranking_order
    ]


def exact_top_k_mask(
    score,
    selection_ratio,
):
    """Construct an exact top-k binary selection mask."""

    selected_count = int(
        np.ceil(
            selection_ratio
            * score.size
        )
    )

    selected_indices = exact_top_k_order(
        score,
        selected_count,
    )

    selected_mask = np.zeros(
        score.size,
        dtype=bool,
    )

    selected_mask[
        selected_indices
    ] = True

    return selected_mask


def evaluate_score_at_budgets(
    score,
    scenario_name,
    variant_name,
    section_name,
):
    """
    Evaluate all budgets using one exact ranking up to the
    largest required budget.
    """

    maximum_budget_ratio = max(
        BUDGETS.values()
    )

    maximum_selected_count = int(
        np.ceil(
            maximum_budget_ratio
            * N_SAMPLES
        )
    )

    top_order = exact_top_k_order(
        score,
        maximum_selected_count,
    )

    cumulative_injected = np.cumsum(
        injected_mask[
            top_order
        ],
        dtype=np.int64,
    )

    rows = []

    for budget_name, budget_ratio in (
        BUDGETS.items()
    ):

        selected_count = int(
            np.ceil(
                budget_ratio
                * N_SAMPLES
            )
        )

        detected_count = int(
            cumulative_injected[
                selected_count - 1
            ]
        )

        detection_rate = (
            detected_count
            / N_INJECTED
            * 100.0
        )

        rows.append(
            {
                "Section": section_name,
                "Scenario key": scenario_name,
                "Scenario": SCENARIOS[
                    scenario_name
                ],
                "Variant": variant_name,
                "Budget": budget_name,
                "Budget ratio": budget_ratio,
                "Selected pixels": selected_count,
                "Injected pixels": N_INJECTED,
                "Detected injected pixels":
                    detected_count,
                "Detection rate (%)":
                    detection_rate,
            }
        )

    return rows


# ============================================================
# 11. Variant specification helper
# ============================================================

def make_specification(
    description,
    base_keys,
    use_spatial=False,
    use_context=False,
    use_reliability=False,
    family=None,
    fixed_spatial_bounds=None,
):
    """Create one clear ablation specification."""

    if family == "full":

        base_weight = (
            FULL_BASE_WEIGHT
        )

        spatial_weight = (
            FULL_SPATIAL_WEIGHT
        )

    elif family == "pure":

        base_weight = (
            PURE_BASE_WEIGHT
        )

        spatial_weight = (
            PURE_SPATIAL_WEIGHT
        )

    else:

        base_weight = 1.0
        spatial_weight = 0.0

    return {
        "description": description,
        "base_keys": list(
            base_keys
        ),
        "use_spatial": bool(
            use_spatial
        ),
        "use_context": bool(
            use_context
        ),
        "use_reliability": bool(
            use_reliability
        ),
        "family": family,
        "base_weight": float(
            base_weight
        ),
        "spatial_weight": float(
            spatial_weight
        ),
        "fixed_spatial_bounds":
            fixed_spatial_bounds,
    }




In [ ]:
# ============================================================
# 12. Subsection 1:
# Contribution of Representation and Detector Components
# ============================================================

REPRESENTATION_VARIANTS = OrderedDict(
    [
        (
            "Full base",
            make_specification(
                description=(
                    "AE plus all six feature- and "
                    "embedding-space detector scores"
                ),
                base_keys=FULL_BASE_KEYS,
            ),
        ),
        (
            "AE only",
            make_specification(
                description=(
                    "Multi-view autoencoder "
                    "reconstruction score only"
                ),
                base_keys=[
                    "ae",
                ],
            ),
        ),
        (
            "Feature-space detectors",
            make_specification(
                description=(
                    "IF, LOF, and MD in the original "
                    "feature space"
                ),
                base_keys=FEATURE_DETECTOR_KEYS,
            ),
        ),
        (
            "Embedding-space detectors",
            make_specification(
                description=(
                    "IF, LOF, and MD in the learned "
                    "embedding space"
                ),
                base_keys=EMBEDDING_DETECTOR_KEYS,
            ),
        ),
        (
            "Classical detectors",
            make_specification(
                description=(
                    "All six feature- and "
                    "embedding-space detector scores; "
                    "AE excluded"
                ),
                base_keys=CLASSICAL_DETECTOR_KEYS,
            ),
        ),
        (
            "Full base without feature space",
            make_specification(
                description=(
                    "AE plus embedding-space IF, LOF, "
                    "and MD"
                ),
                base_keys=(
                    ["ae"]
                    + EMBEDDING_DETECTOR_KEYS
                ),
            ),
        ),
        (
            "Full base without embedding space",
            make_specification(
                description=(
                    "AE plus feature-space IF, LOF, "
                    "and MD"
                ),
                base_keys=(
                    ["ae"]
                    + FEATURE_DETECTOR_KEYS
                ),
            ),
        ),
        (
            "Full base without IF",
            make_specification(
                description=(
                    "Full base after removing both "
                    "feature- and embedding-space IF"
                ),
                base_keys=[
                    score_key
                    for score_key in FULL_BASE_KEYS
                    if score_key
                    not in IF_FAMILY_KEYS
                ],
            ),
        ),
        (
            "Full base without LOF",
            make_specification(
                description=(
                    "Full base after removing both "
                    "feature- and embedding-space LOF"
                ),
                base_keys=[
                    score_key
                    for score_key in FULL_BASE_KEYS
                    if score_key
                    not in LOF_FAMILY_KEYS
                ],
            ),
        ),
        (
            "Full base without MD",
            make_specification(
                description=(
                    "Full base after removing both "
                    "feature- and embedding-space MD"
                ),
                base_keys=[
                    score_key
                    for score_key in FULL_BASE_KEYS
                    if score_key
                    not in MD_FAMILY_KEYS
                ],
            ),
        ),
    ]
)




In [ ]:
# ============================================================
# 13. Subsection 2:
# Contribution of Spatial, Contextual, and Reliability Modules
# ============================================================

MODULE_VARIANTS = OrderedDict(
    [
        (
            "Full base",
            REPRESENTATION_VARIANTS[
                "Full base"
            ],
        ),
        (
            "Full without spatial",
            make_specification(
                description=(
                    "Full base with context and "
                    "reliability, but without the "
                    "spatial score"
                ),
                base_keys=FULL_BASE_KEYS,
                use_spatial=False,
                use_context=True,
                use_reliability=True,
                family="full",
            ),
        ),
        (
            "Full without context",
            make_specification(
                description=(
                    "Full spatial fusion with "
                    "reliability estimated without "
                    "the context score"
                ),
                base_keys=FULL_BASE_KEYS,
                use_spatial=True,
                use_context=False,
                use_reliability=True,
                family="full",
                fixed_spatial_bounds=(
                    FULL_SPATIAL_P1,
                    FULL_SPATIAL_P99,
                ),
            ),
        ),
        (
            "Full without reliability",
            make_specification(
                description=(
                    "Full spatially refined score "
                    "before reliability weighting; "
                    "context has no direct fusion path"
                ),
                base_keys=FULL_BASE_KEYS,
                use_spatial=True,
                use_context=True,
                use_reliability=False,
                family="full",
                fixed_spatial_bounds=(
                    FULL_SPATIAL_P1,
                    FULL_SPATIAL_P99,
                ),
            ),
        ),
        (
            "Pure base",
            make_specification(
                description=(
                    "AE reconstruction score before "
                    "spatial, context, and reliability "
                    "modules"
                ),
                base_keys=[
                    "ae",
                ],
            ),
        ),
        (
            "Pure without spatial",
            make_specification(
                description=(
                    "AE with context and reliability, "
                    "but without the spatial score"
                ),
                base_keys=[
                    "ae",
                ],
                use_spatial=False,
                use_context=True,
                use_reliability=True,
                family="pure",
            ),
        ),
        (
            "Pure without context",
            make_specification(
                description=(
                    "Pure spatial fusion with "
                    "reliability estimated without "
                    "the context score"
                ),
                base_keys=[
                    "ae",
                ],
                use_spatial=True,
                use_context=False,
                use_reliability=True,
                family="pure",
                fixed_spatial_bounds=(
                    PURE_SPATIAL_P1,
                    PURE_SPATIAL_P99,
                ),
            ),
        ),
        (
            "Pure without reliability",
            make_specification(
                description=(
                    "Pure spatially refined score "
                    "before reliability weighting; "
                    "context has no direct fusion path"
                ),
                base_keys=[
                    "ae",
                ],
                use_spatial=True,
                use_context=True,
                use_reliability=False,
                family="pure",
                fixed_spatial_bounds=(
                    PURE_SPATIAL_P1,
                    PURE_SPATIAL_P99,
                ),
            ),
        ),
    ]
)


# Original complete variants use the already validated raw scores.
REFERENCE_VARIANTS = [
    "Full",
    "Pure",
]




In [ ]:
# ============================================================
# 14. Prepare unique computed variants and descriptions
# ============================================================

ALL_COMPUTED_VARIANTS = OrderedDict()

for variant_name, specification in (
    REPRESENTATION_VARIANTS.items()
):
    ALL_COMPUTED_VARIANTS[
        variant_name
    ] = specification

for variant_name, specification in (
    MODULE_VARIANTS.items()
):
    ALL_COMPUTED_VARIANTS[
        variant_name
    ] = specification


variant_definition_rows = []

for variant_name, specification in (
    REPRESENTATION_VARIANTS.items()
):

    variant_definition_rows.append(
        {
            "Paper subsection":
                "Representation and detectors",
            "Variant": variant_name,
            "Base signals": ", ".join(
                specification["base_keys"]
            ),
            "Spatial included":
                specification["use_spatial"],
            "Context included":
                specification["use_context"],
            "Reliability included":
                specification["use_reliability"],
            "Description":
                specification["description"],
        }
    )


for variant_name, specification in (
    MODULE_VARIANTS.items()
):

    variant_definition_rows.append(
        {
            "Paper subsection":
                "Spatial, context, and reliability",
            "Variant": variant_name,
            "Base signals": ", ".join(
                specification["base_keys"]
            ),
            "Spatial included":
                specification["use_spatial"],
            "Context included":
                specification["use_context"],
            "Reliability included":
                specification["use_reliability"],
            "Description":
                specification["description"],
        }
    )


variant_definition_rows.extend(
    [
        {
            "Paper subsection":
                "Spatial, context, and reliability",
            "Variant": "Full",
            "Base signals":
                ", ".join(FULL_BASE_KEYS),
            "Spatial included": True,
            "Context included": True,
            "Reliability included": True,
            "Description": (
                "Original validated Full raw final "
                "ranking score"
            ),
        },
        {
            "Paper subsection":
                "Spatial, context, and reliability",
            "Variant": "Pure",
            "Base signals": "ae",
            "Spatial included": True,
            "Context included": True,
            "Reliability included": True,
            "Description": (
                "Original validated Pure raw final "
                "ranking score"
            ),
        },
    ]
)

variant_definition_df = pd.DataFrame(
    variant_definition_rows
).drop_duplicates(
    subset=[
        "Paper subsection",
        "Variant",
    ]
)


# ============================================================
# 15. Derive clean calibration for every computed ablation
# ============================================================

calibrations = OrderedDict()
clean_variant_masks = OrderedDict()
calibration_rows = []
clean_score_summary_rows = []

TOP2_RATIO = 0.02

for variant_name, specification in (
    ALL_COMPUTED_VARIANTS.items()
):

    print(
        f"Calibrating clean variant: {variant_name}"
    )

    clean_variant_score, calibration = (
        construct_variant_score(
            clean_scores,
            clean_labels,
            specification,
            calibration=None,
        )
    )

    calibrations[
        variant_name
    ] = calibration

    clean_variant_masks[
        variant_name
    ] = exact_top_k_mask(
        clean_variant_score,
        TOP2_RATIO,
    )

    calibration_rows.append(
        {
            "Variant": variant_name,
            **calibration,
        }
    )

    clean_score_summary_rows.append(
        {
            "Variant": variant_name,
            "Minimum score": float(
                clean_variant_score.min()
            ),
            "Maximum score": float(
                clean_variant_score.max()
            ),
            "Mean score": float(
                np.mean(
                    clean_variant_score,
                    dtype=np.float64,
                )
            ),
            "Median score": float(
                np.median(
                    clean_variant_score
                )
            ),
            "98th percentile": float(
                np.percentile(
                    clean_variant_score,
                    98,
                )
            ),
        }
    )


calibration_df = pd.DataFrame(
    calibration_rows
)

clean_score_summary_df = pd.DataFrame(
    clean_score_summary_rows
)


# ============================================================
# 16. Load exact original Full and Pure clean references
# ============================================================

full_clean_reference_score = load_vector(
    FULL_CLEAN_PRECLIP_PATH,
    expected_length=N_SAMPLES,
    name="clean Full pre-clipping score",
    mmap=True,
)

pure_clean_reference_score = load_vector(
    PURE_CLEAN_PRECLIP_PATH,
    expected_length=N_SAMPLES,
    name="clean Pure pre-clipping score",
    mmap=True,
)

clean_variant_masks[
    "Full"
] = exact_top_k_mask(
    full_clean_reference_score,
    TOP2_RATIO,
)

clean_variant_masks[
    "Pure"
] = exact_top_k_mask(
    pure_clean_reference_score,
    TOP2_RATIO,
)


# ============================================================
# 17. Validate clean Full and Pure score reconstruction
# ============================================================

full_base_clean = average_scores(
    clean_scores,
    FULL_BASE_KEYS,
)

full_spatial_raw_clean = (
    FULL_BASE_WEIGHT
    * full_base_clean
    + FULL_SPATIAL_WEIGHT
    * np.asarray(
        clean_scores["spatial"],
        dtype=np.float32,
    )
).astype(
    np.float32,
    copy=False,
)

full_spatial_normalized_clean = (
    normalize_using_bounds(
        full_spatial_raw_clean,
        FULL_SPATIAL_P1,
        FULL_SPATIAL_P99,
    )
)


pure_spatial_raw_clean = (
    PURE_BASE_WEIGHT
    * np.asarray(
        clean_scores["ae"],
        dtype=np.float32,
    )
    + PURE_SPATIAL_WEIGHT
    * np.asarray(
        clean_scores["pure_spatial"],
        dtype=np.float32,
    )
).astype(
    np.float32,
    copy=False,
)

pure_spatial_normalized_clean = (
    normalize_using_bounds(
        pure_spatial_raw_clean,
        PURE_SPATIAL_P1,
        PURE_SPATIAL_P99,
    )
)


with np.load(
    CLEAN_RELIABILITY_PATH
) as reliability_archive:

    saved_full_reliability = np.asarray(
        reliability_archive[
            "full_reliability"
        ]
    ).reshape(-1)

    saved_pure_reliability = np.asarray(
        reliability_archive[
            "pure_reliability"
        ]
    ).reshape(-1)


if saved_full_reliability.size != N_SAMPLES:
    raise ValueError(
        "Saved clean Full reliability has invalid length."
    )

if saved_pure_reliability.size != N_SAMPLES:
    raise ValueError(
        "Saved clean Pure reliability has invalid length."
    )


reconstructed_full_clean_score = (
    full_spatial_normalized_clean
    * saved_full_reliability
).astype(
    np.float32,
    copy=False,
)

reconstructed_pure_clean_score = (
    pure_spatial_normalized_clean
    * saved_pure_reliability
).astype(
    np.float32,
    copy=False,
)


full_clean_max_difference = float(
    np.max(
        np.abs(
            reconstructed_full_clean_score
            - full_clean_reference_score
        )
    )
)

pure_clean_max_difference = float(
    np.max(
        np.abs(
            reconstructed_pure_clean_score
            - pure_clean_reference_score
        )
    )
)




In [ ]:
# ============================================================
# 18. Original-data top-2% overlap analysis
# ============================================================

def mask_overlap_summary(
    variant_mask,
    reference_mask,
):
    """Calculate clean top-2% selection overlap."""

    intersection = int(
        np.logical_and(
            variant_mask,
            reference_mask,
        ).sum()
    )

    union = int(
        np.logical_or(
            variant_mask,
            reference_mask,
        ).sum()
    )

    jaccard = (
        intersection / union
        if union > 0
        else np.nan
    )

    reference_count = int(
        reference_mask.sum()
    )

    replacement_rate = (
        100.0
        * (
            reference_count
            - intersection
        )
        / reference_count
    )

    return {
        "Intersection": intersection,
        "Union": union,
        "Jaccard similarity": jaccard,
        "Replacement rate (%)":
            replacement_rate,
    }


clean_overlap_rows = []

full_base_reference_mask = (
    clean_variant_masks[
        "Full base"
    ]
)

for variant_name in (
    REPRESENTATION_VARIANTS.keys()
):

    overlap = mask_overlap_summary(
        clean_variant_masks[
            variant_name
        ],
        full_base_reference_mask,
    )

    clean_overlap_rows.append(
        {
            "Section":
                "Representation and detectors",
            "Variant": variant_name,
            "Reference": "Full base",
            **overlap,
        }
    )


for variant_name in [
    "Full",
    "Full base",
    "Full without spatial",
    "Full without context",
    "Full without reliability",
]:

    overlap = mask_overlap_summary(
        clean_variant_masks[
            variant_name
        ],
        clean_variant_masks[
            "Full"
        ],
    )

    clean_overlap_rows.append(
        {
            "Section":
                "Spatial, context, and reliability",
            "Variant": variant_name,
            "Reference": "Full",
            **overlap,
        }
    )


for variant_name in [
    "Pure",
    "Pure base",
    "Pure without spatial",
    "Pure without context",
    "Pure without reliability",
]:

    overlap = mask_overlap_summary(
        clean_variant_masks[
            variant_name
        ],
        clean_variant_masks[
            "Pure"
        ],
    )

    clean_overlap_rows.append(
        {
            "Section":
                "Spatial, context, and reliability",
            "Variant": variant_name,
            "Reference": "Pure",
            **overlap,
        }
    )


clean_overlap_df = pd.DataFrame(
    clean_overlap_rows
)




In [ ]:
# ============================================================
# 19. Controlled-injection ablation evaluation
# ============================================================

all_detection_rows = []
all_threshold_rows = []

for scenario_key, scenario_display in (
    SCENARIOS.items()
):

    print(
        "\n"
        + "=" * 70
    )
    print(
        f"Processing scenario: {scenario_display}"
    )
    print(
        "=" * 70
    )

    scenario_dir = (
        INJECTED_RESULTS_DIR
        / scenario_key
    )

    scenario_scores = load_component_scores(
        scenario_dir,
        expected_length=N_SAMPLES,
    )

    (
        scenario_labels,
        scenario_thresholds,
    ) = load_internal_signal_labels(
        scenario_dir,
        scenario_scores,
        N_SAMPLES,
    )

    scenario_thresholds.insert(
        0,
        "Scenario",
        scenario_display,
    )

    all_threshold_rows.append(
        scenario_thresholds
    )

    # --------------------------------------------------------
    # Subsection 1:
    # Representation and detector components
    # --------------------------------------------------------

    for variant_name, specification in (
        REPRESENTATION_VARIANTS.items()
    ):

        print(
            "Representation variant:",
            variant_name,
        )

        variant_score, _ = (
            construct_variant_score(
                scenario_scores,
                scenario_labels,
                specification,
                calibration=calibrations[
                    variant_name
                ],
            )
        )

        all_detection_rows.extend(
            evaluate_score_at_budgets(
                variant_score,
                scenario_key,
                variant_name,
                (
                    "Representation and "
                    "detector components"
                ),
            )
        )

    # --------------------------------------------------------
    # Original Full and Pure references
    # --------------------------------------------------------

    full_reference_score = load_vector(
        scenario_dir
        / "full_final_anomaly_score_raw.npy",
        expected_length=N_SAMPLES,
        name=(
            f"{scenario_key} Full raw final score"
        ),
        mmap=True,
    )

    pure_reference_score = load_vector(
        scenario_dir
        / "pure_final_anomaly_score_raw.npy",
        expected_length=N_SAMPLES,
        name=(
            f"{scenario_key} Pure raw final score"
        ),
        mmap=True,
    )

    all_detection_rows.extend(
        evaluate_score_at_budgets(
            full_reference_score,
            scenario_key,
            "Full",
            (
                "Spatial, contextual, and "
                "reliability modules"
            ),
        )
    )

    all_detection_rows.extend(
        evaluate_score_at_budgets(
            pure_reference_score,
            scenario_key,
            "Pure",
            (
                "Spatial, contextual, and "
                "reliability modules"
            ),
        )
    )

    # --------------------------------------------------------
    # Subsection 2:
    # Spatial, contextual, and reliability modules
    # --------------------------------------------------------

    for variant_name, specification in (
        MODULE_VARIANTS.items()
    ):

        print(
            "Module variant:",
            variant_name,
        )

        variant_score, _ = (
            construct_variant_score(
                scenario_scores,
                scenario_labels,
                specification,
                calibration=calibrations[
                    variant_name
                ],
            )
        )

        all_detection_rows.extend(
            evaluate_score_at_budgets(
                variant_score,
                scenario_key,
                variant_name,
                (
                    "Spatial, contextual, and "
                    "reliability modules"
                ),
            )
        )


detection_results_df = pd.DataFrame(
    all_detection_rows
)

signal_thresholds_df = pd.concat(
    all_threshold_rows,
    ignore_index=True,
)




In [ ]:
# ============================================================
# 20. Create paper-oriented summary tables
# ============================================================

REPRESENTATION_ORDER = list(
    REPRESENTATION_VARIANTS.keys()
)

MODULE_ORDER = [
    "Full",
    "Full base",
    "Full without spatial",
    "Full without context",
    "Full without reliability",
    "Pure",
    "Pure base",
    "Pure without spatial",
    "Pure without context",
    "Pure without reliability",
]

SCENARIO_COLUMN_ORDER = list(
    SCENARIOS.values()
)


def create_top2_scenario_table(
    results,
    section_name,
    variant_order,
):
    """Create scenario-wise top-2% detection table."""

    selected_rows = results[
        (
            results["Section"]
            == section_name
        )
        & (
            results["Budget"]
            == "2.00%"
        )
    ].copy()

    table = selected_rows.pivot(
        index="Variant",
        columns="Scenario",
        values="Detection rate (%)",
    )

    table = table.reindex(
        index=variant_order,
        columns=SCENARIO_COLUMN_ORDER,
    )

    table["Mean"] = table.mean(
        axis=1
    )

    table = table.reset_index()

    return table


def create_budget_mean_table(
    results,
    section_name,
    variant_order,
):
    """Create mean detection rate over all five scenarios."""

    selected_rows = results[
        results["Section"]
        == section_name
    ].copy()

    mean_rows = (
        selected_rows
        .groupby(
            [
                "Variant",
                "Budget",
            ],
            as_index=False,
        )[
            "Detection rate (%)"
        ]
        .mean()
    )

    table = mean_rows.pivot(
        index="Variant",
        columns="Budget",
        values="Detection rate (%)",
    )

    table = table.reindex(
        index=variant_order,
        columns=list(
            BUDGETS.keys()
        ),
    )

    table = table.reset_index()

    return table


representation_section_name = (
    "Representation and detector components"
)

module_section_name = (
    "Spatial, contextual, and reliability modules"
)


representation_top2_df = (
    create_top2_scenario_table(
        detection_results_df,
        representation_section_name,
        REPRESENTATION_ORDER,
    )
)

representation_budget_mean_df = (
    create_budget_mean_table(
        detection_results_df,
        representation_section_name,
        REPRESENTATION_ORDER,
    )
)


module_top2_df = (
    create_top2_scenario_table(
        detection_results_df,
        module_section_name,
        MODULE_ORDER,
    )
)

module_budget_mean_df = (
    create_budget_mean_table(
        detection_results_df,
        module_section_name,
        MODULE_ORDER,
    )
)


# Delta against the Full base representation
full_base_mean = float(
    representation_top2_df.loc[
        representation_top2_df[
            "Variant"
        ] == "Full base",
        "Mean",
    ].iloc[0]
)

representation_top2_df[
    "Delta vs Full base (pp)"
] = (
    representation_top2_df[
        "Mean"
    ]
    - full_base_mean
)


# Module deltas use the appropriate complete variant
full_final_mean = float(
    module_top2_df.loc[
        module_top2_df[
            "Variant"
        ] == "Full",
        "Mean",
    ].iloc[0]
)

pure_final_mean = float(
    module_top2_df.loc[
        module_top2_df[
            "Variant"
        ] == "Pure",
        "Mean",
    ].iloc[0]
)


module_reference_names = []
module_deltas = []

for _, current_row in (
    module_top2_df.iterrows()
):

    current_variant = str(
        current_row["Variant"]
    )

    if current_variant.startswith(
        "Pure"
    ):

        reference_name = "Pure"
        reference_mean = pure_final_mean

    else:

        reference_name = "Full"
        reference_mean = full_final_mean

    module_reference_names.append(
        reference_name
    )

    module_deltas.append(
        float(
            current_row["Mean"]
            - reference_mean
        )
    )


module_top2_df[
    "Reference"
] = module_reference_names

module_top2_df[
    "Delta vs reference (pp)"
] = module_deltas


# ============================================================
# 21. Validation summary
# ============================================================

validation_rows = [
    {
        "Check": "Dataset length",
        "Passed": (
            N_SAMPLES
            == 4_290_958
        ),
        "Details": (
            f"{N_SAMPLES:,} samples"
        ),
    },
    {
        "Check": "Injected sample count",
        "Passed": (
            N_INJECTED
            == 4_290
        ),
        "Details": (
            f"{N_INJECTED:,} injected samples"
        ),
    },
    {
        "Check": "Injected indices unique",
        "Passed": (
            np.unique(
                inject_indices
            ).size
            == N_INJECTED
        ),
        "Details": (
            "No duplicated injected indices"
        ),
    },
    {
        "Check": "Full clean score reconstruction",
        "Passed": (
            full_clean_max_difference
            <= 1e-6
        ),
        "Details": (
            "Maximum absolute difference = "
            f"{full_clean_max_difference:.10g}"
        ),
    },
    {
        "Check": "Pure clean score reconstruction",
        "Passed": (
            pure_clean_max_difference
            <= 1e-6
        ),
        "Details": (
            "Maximum absolute difference = "
            f"{pure_clean_max_difference:.10g}"
        ),
    },
    {
        "Check": "All final evaluation counts exact",
        "Passed": bool(
            (
                detection_results_df[
                    "Selected pixels"
                ]
                == detection_results_df[
                    "Budget ratio"
                ].apply(
                    lambda ratio: int(
                        np.ceil(
                            ratio
                            * N_SAMPLES
                        )
                    )
                )
            ).all()
        ),
        "Details": (
            "Exact deterministic top-k selection"
        ),
    },
    {
        "Check": "Detection rates within valid range",
        "Passed": bool(
            detection_results_df[
                "Detection rate (%)"
            ].between(
                0.0,
                100.0,
            ).all()
        ),
        "Details": (
            "All detection rates are in [0,100]"
        ),
    },
    {
        "Check": "All five scenarios evaluated",
        "Passed": (
            detection_results_df[
                "Scenario key"
            ].nunique()
            == 5
        ),
        "Details": (
            ", ".join(
                detection_results_df[
                    "Scenario key"
                ].drop_duplicates()
            )
        ),
    },
    {
        "Check": "All five budgets evaluated",
        "Passed": (
            detection_results_df[
                "Budget"
            ].nunique()
            == 5
        ),
        "Details": (
            ", ".join(
                detection_results_df[
                    "Budget"
                ].drop_duplicates()
            )
        ),
    },
]

validation_df = pd.DataFrame(
    validation_rows
)


print("\nValidation summary:")
print(
    validation_df.to_string(
        index=False
    )
)

if not validation_df[
    "Passed"
].all():

    print(
        "\nWARNING: At least one validation check failed. "
        "Review the Validation sheet before using the results."
    )


# ============================================================
# 22. Experimental settings, ablation scope, and paper guidance
# ============================================================

POST_TRAINING_SCOPE_TEXT = (
    "This study is a post-training score-level ablation. "
    "All trained autoencoders, outlier detectors, preprocessing "
    "transformations, fitted model parameters, and normalized "
    "component anomaly scores are held fixed. The ablations "
    "evaluate the contribution of saved component scores and "
    "downstream score-fusion modules rather than the effect of "
    "retraining a modified architecture."
)

REPRESENTATION_SCOPE_TEXT = (
    "Representation and detector configurations are evaluated "
    "at the base-score level before spatial, contextual, and "
    "reliability refinement. Detector-removal configurations "
    "therefore measure each detector family's contribution to "
    "the initial fused anomaly representation and are not "
    "end-to-end variants of the complete Full model."
)

FULL_BASE_WITHOUT_IF_DEFINITION = (
    "Full base without IF removes feature-space IF and "
    "embedding-space IF from the initial Full base-score fusion. "
    "It retains AE, feature-space LOF and MD, and embedding-space "
    "LOF and MD. Spatial inconsistency, contextual consistency, "
    "reliability, and the final Full score are not recomputed. "
    "The result therefore measures IF's contribution only at "
    "the Full base-score level."
)

MODULE_SCOPE_TEXT = (
    "The module ablations use the saved post-training component "
    "scores. Removing spatial excludes the spatial contribution "
    "from the evaluated score construction; removing context "
    "excludes context from reliability estimation; and removing "
    "reliability ranks the spatially refined score before the "
    "reliability-weighted final scoring stage."
)


settings_df = pd.DataFrame(
    [
        {
            "Setting": "Dataset",
            "Value": (
                "Original clean dataset plus the reference "
                "0.1% controlled-injection datasets"
            ),
        },
        {
            "Setting": "Model seed",
            "Value": SEED,
        },
        {
            "Setting": "Injection seed",
            "Value": SEED,
        },
        {
            "Setting": "Number of pixels",
            "Value": N_SAMPLES,
        },
        {
            "Setting": "Injected pixels",
            "Value": N_INJECTED,
        },
        {
            "Setting": "Actual injection ratio",
            "Value": ACTUAL_INJECTION_RATIO,
        },
        {
            "Setting": "Injection scenarios",
            "Value": ", ".join(
                SCENARIOS.values()
            ),
        },
        {
            "Setting": "Internal anomaly-flag rule",
            "Value": (
                "Deterministic exact top-2% selection for "
                "spatial/context signals; saved exact detector "
                "labels reused where available"
            ),
        },
        {
            "Setting": "Final ranking rule",
            "Value": (
                "Deterministic exact top-k ranking using the "
                "evaluated score before final output clipping"
            ),
        },
        {
            "Setting": "Final evaluation budgets",
            "Value": ", ".join(
                BUDGETS.keys()
            ),
        },
        {
            "Setting": "Ablation type",
            "Value": POST_TRAINING_SCOPE_TEXT,
        },
        {
            "Setting": "Representation-ablation scope",
            "Value": REPRESENTATION_SCOPE_TEXT,
        },
        {
            "Setting": "Full base without IF definition",
            "Value": FULL_BASE_WITHOUT_IF_DEFINITION,
        },
        {
            "Setting": "Module-ablation scope",
            "Value": MODULE_SCOPE_TEXT,
        },
        {
            "Setting": "Models retrained",
            "Value": False,
        },
        {
            "Setting": "Primary evaluation metric",
            "Value": (
                "Injected-anomaly detection rate (%)"
            ),
        },
        {
            "Setting": "Secondary diagnostic",
            "Value": (
                "Clean-data exact top-2% selection Jaccard "
                "similarity and replacement rate"
            ),
        },
    ]
)


ablation_scope_df = pd.DataFrame(
    [
        {
            "Scope item": "Overall ablation",
            "Definition": POST_TRAINING_SCOPE_TEXT,
        },
        {
            "Scope item": (
                "Representation and detector components"
            ),
            "Definition": REPRESENTATION_SCOPE_TEXT,
        },
        {
            "Scope item": "Full base without IF",
            "Definition": FULL_BASE_WITHOUT_IF_DEFINITION,
        },
        {
            "Scope item": (
                "Spatial, context, and reliability modules"
            ),
            "Definition": MODULE_SCOPE_TEXT,
        },
    ]
)


paper_guidance_df = pd.DataFrame(
    [
        {
            "Paper subsection": (
                "Contribution of Representation and "
                "Detector Components"
            ),
            "Primary table": "Rep Top2",
            "Supporting table": "Rep Budget Means",
            "Interpretation": (
                "Compare Full base with AE-only, feature-space, "
                "embedding-space, classical-detector, "
                "representation-removal, and detector-family "
                "removal configurations. These are base-score "
                "comparisons, not complete Full-model variants."
            ),
        },
        {
            "Paper subsection": (
                "Contribution of Spatial, Contextual, "
                "and Reliability Modules"
            ),
            "Primary table": "Module Top2",
            "Supporting table": "Module Budget Means",
            "Interpretation": (
                "Compare each module-removal configuration with "
                "its complete Full or Pure reference. Interpret "
                "changes as contributions to post-training score "
                "construction and ranking."
            ),
        },
        {
            "Paper subsection": "Both",
            "Primary table": "Clean Overlap",
            "Supporting table": "Clean Score Summary",
            "Interpretation": (
                "Use clean-data overlap to characterize how the "
                "selected anomaly population changes. Do not "
                "interpret overlap as detection accuracy because "
                "complete original-data ground truth is absent."
            ),
        },
    ]
)


# Save an independent text record of the exact scope.
with open(
    OUTPUT_DIR
    / "Ablation_Study_Scope.txt",
    "w",
    encoding="utf-8",
) as scope_file:

    scope_file.write(
        "OVERALL POST-TRAINING SCOPE\n"
        "===========================\n"
        f"{POST_TRAINING_SCOPE_TEXT}\n\n"
        "REPRESENTATION-LEVEL SCOPE\n"
        "==========================\n"
        f"{REPRESENTATION_SCOPE_TEXT}\n\n"
        "FULL BASE WITHOUT IF\n"
        "====================\n"
        f"{FULL_BASE_WITHOUT_IF_DEFINITION}\n\n"
        "MODULE-LEVEL SCOPE\n"
        "==================\n"
        f"{MODULE_SCOPE_TEXT}\n"
    )


# ============================================================
# 23. Save CSV and LaTeX outputs
# ============================================================

detection_results_df.to_csv(
    OUTPUT_DIR
    / "ablation_detection_results_long.csv",
    index=False,
)

representation_top2_df.to_csv(
    OUTPUT_DIR
    / "representation_ablation_top2.csv",
    index=False,
)

module_top2_df.to_csv(
    OUTPUT_DIR
    / "module_ablation_top2.csv",
    index=False,
)


with open(
    OUTPUT_DIR
    / "table_representation_ablation_top2.tex",
    "w",
    encoding="utf-8",
) as latex_file:

    latex_file.write(
        representation_top2_df.to_latex(
            index=False,
            float_format="%.2f",
            escape=True,
        )
    )


with open(
    OUTPUT_DIR
    / "table_module_ablation_top2.tex",
    "w",
    encoding="utf-8",
) as latex_file:

    latex_file.write(
        module_top2_df.to_latex(
            index=False,
            float_format="%.2f",
            escape=True,
        )
    )




In [ ]:
# ============================================================
# 24. Heatmap function
# ============================================================

def save_detection_heatmap(
    table,
    scenario_columns,
    output_path,
):
    """Save a paper-oriented annotated heatmap."""

    plot_table = table.set_index(
        "Variant"
    )[scenario_columns]

    values = plot_table.to_numpy(
        dtype=float
    )

    figure_height = max(
        5.0,
        0.55 * len(plot_table)
    )

    plt.figure(
        figsize=(
            10,
            figure_height,
        )
    )

    image = plt.imshow(
        values,
        aspect="auto",
        vmin=0.0,
        vmax=100.0,
        interpolation="nearest",
    )

    plt.colorbar(
        image,
        label=(
            "Injected-anomaly detection rate (%)"
        ),
    )

    plt.xticks(
        np.arange(
            len(scenario_columns)
        ),
        scenario_columns,
        rotation=25,
        ha="right",
    )

    plt.yticks(
        np.arange(
            len(plot_table.index)
        ),
        plot_table.index,
    )

    for row_index in range(
        values.shape[0]
    ):

        for column_index in range(
            values.shape[1]
        ):
            current_value = values[
                row_index,
                column_index,
            ]
            annotation_color = (
                "white"
                if current_value < 50.0
                else "black"
            )

            plt.text(
                column_index,
                row_index,
                f"{current_value:.1f}",
                ha="center",
                va="center",
                color=annotation_color,
            )

    plt.xlabel(
        "Synthetic anomaly scenario"
    )

    plt.ylabel(
        "Ablation configuration"
    )

    plt.tight_layout()

    plt.savefig(
        output_path,
        dpi=300,
        bbox_inches="tight",
    )

    plt.close()


save_detection_heatmap(
    representation_top2_df,
    SCENARIO_COLUMN_ORDER,
    OUTPUT_DIR
    / "Representation_Ablation_Top2_Heatmap.png",
)

save_detection_heatmap(
    module_top2_df,
    SCENARIO_COLUMN_ORDER,
    OUTPUT_DIR
    / "Module_Ablation_Top2_Heatmap.png",
)




In [ ]:
# ============================================================
# 25. Figures: mean detection rate across budgets
# ============================================================

def save_budget_mean_figure(
    table,
    variant_order,
    output_path,
):
    """
    Plot mean detection rate across the five synthetic scenarios
    for a selected group of ablation configurations.
    """

    plotted_table = (
        table
        .set_index(
            "Variant"
        )
        .reindex(
            variant_order
        )
    )

    budget_names = list(
        BUDGETS.keys()
    )

    missing_variants = [
        variant_name
        for variant_name in variant_order
        if variant_name
        not in plotted_table.dropna(
            how="all"
        ).index
    ]

    if missing_variants:
        raise ValueError(
            "The following requested figure variants "
            "were not found in the table: "
            + ", ".join(
                missing_variants
            )
        )

    x_positions = np.arange(
        len(budget_names)
    )

    plt.figure(
        figsize=(
            9,
            5.5,
        )
    )

    for variant_name in variant_order:

        variant_values = (
            plotted_table.loc[
                variant_name,
                budget_names,
            ]
            .to_numpy(
                dtype=float
            )
        )

        plt.plot(
            x_positions,
            variant_values,
            marker="o",
            label=variant_name,
        )

    plt.xticks(
        x_positions,
        budget_names,
    )

    plt.xlabel(
        "Exact final selection budget"
    )

    plt.ylabel(
        "Mean injected-anomaly detection rate (%)"
    )

    # Use the same vertical scale in every ablation figure.
    plt.ylim(
        0,
        102,
    )

    plt.grid(
        axis="y",
        alpha=0.3,
    )

    plt.legend(
        bbox_to_anchor=(
            1.02,
            1.0,
        ),
        loc="upper left",
    )

    plt.tight_layout()

    plt.savefig(
        output_path,
        dpi=300,
        bbox_inches="tight",
    )

    plt.close()


# ------------------------------------------------------------
# 25A. Core representation comparison
# ------------------------------------------------------------

REPRESENTATION_CORE_FIGURE_ORDER = [
    "Full base",
    "AE only",
    "Feature-space detectors",
    "Embedding-space detectors",
    "Classical detectors",
]

save_budget_mean_figure(
    representation_budget_mean_df,
    REPRESENTATION_CORE_FIGURE_ORDER,
    OUTPUT_DIR
    / "Representation_Core_Budget_Sensitivity.png",
)


# ------------------------------------------------------------
# 25B. Representation-space and detector removal
# ------------------------------------------------------------

DETECTOR_REMOVAL_FIGURE_ORDER = [
    "Full base",
    "Full base without feature space",
    "Full base without embedding space",
    "Full base without IF",
    "Full base without LOF",
    "Full base without MD",
]

save_budget_mean_figure(
    representation_budget_mean_df,
    DETECTOR_REMOVAL_FIGURE_ORDER,
    OUTPUT_DIR
    / "Representation_Removal_Budget_Sensitivity.png",
)


# ------------------------------------------------------------
# 25C. Full module ablations
# ------------------------------------------------------------

FULL_MODULE_FIGURE_ORDER = [
    "Full",
    "Full base",
    "Full without spatial",
    "Full without context",
    "Full without reliability",
]

save_budget_mean_figure(
    module_budget_mean_df,
    FULL_MODULE_FIGURE_ORDER,
    OUTPUT_DIR
    / "Full_Module_Budget_Sensitivity.png",
)


# ------------------------------------------------------------
# 25D. Pure module ablations
# ------------------------------------------------------------

PURE_MODULE_FIGURE_ORDER = [
    "Pure",
    "Pure base",
    "Pure without spatial",
    "Pure without context",
    "Pure without reliability",
]

save_budget_mean_figure(
    module_budget_mean_df,
    PURE_MODULE_FIGURE_ORDER,
    OUTPUT_DIR
    / "Pure_Module_Budget_Sensitivity.png",
)

In [ ]:
# ============================================================
# 26. Save complete Excel workbook
# ============================================================

excel_output_path = (
    OUTPUT_DIR
    / "Ablation_Study_Results.xlsx"
)

with pd.ExcelWriter(
    excel_output_path,
    engine="openpyxl",
) as writer:

    validation_df.to_excel(
        writer,
        sheet_name="Validation",
        index=False,
    )

    settings_df.to_excel(
        writer,
        sheet_name="Settings",
        index=False,
    )

    ablation_scope_df.to_excel(
        writer,
        sheet_name="Ablation Scope",
        index=False,
    )

    variant_definition_df.to_excel(
        writer,
        sheet_name="Variants",
        index=False,
    )

    calibration_df.to_excel(
        writer,
        sheet_name="Calibration",
        index=False,
    )

    signal_thresholds_df.to_excel(
        writer,
        sheet_name="Signal Thresholds",
        index=False,
    )

    detection_results_df.to_excel(
        writer,
        sheet_name="Long Results",
        index=False,
    )

    representation_top2_df.to_excel(
        writer,
        sheet_name="Rep Top2",
        index=False,
    )

    representation_budget_mean_df.to_excel(
        writer,
        sheet_name="Rep Budget Means",
        index=False,
    )

    module_top2_df.to_excel(
        writer,
        sheet_name="Module Top2",
        index=False,
    )

    module_budget_mean_df.to_excel(
        writer,
        sheet_name="Module Budget Means",
        index=False,
    )

    clean_overlap_df.to_excel(
        writer,
        sheet_name="Clean Overlap",
        index=False,
    )

    clean_score_summary_df.to_excel(
        writer,
        sheet_name="Clean Score Summary",
        index=False,
    )

    clean_threshold_df.to_excel(
        writer,
        sheet_name="Clean Thresholds",
        index=False,
    )

    paper_guidance_df.to_excel(
        writer,
        sheet_name="Paper Guide",
        index=False,
    )


# ============================================================
# 27. Format Excel workbook
# ============================================================

workbook = load_workbook(
    excel_output_path
)

for worksheet in workbook.worksheets:

    worksheet.freeze_panes = "A2"
    worksheet.auto_filter.ref = (
        worksheet.dimensions
    )

    for cell in worksheet[1]:

        cell.font = Font(
            bold=True
        )

        cell.alignment = Alignment(
            horizontal="center",
            vertical="center",
            wrap_text=True,
        )

    for row in worksheet.iter_rows(
        min_row=2
    ):

        for cell in row:

            cell.alignment = Alignment(
                vertical="top",
                wrap_text=True,
            )

    for column_cells in (
        worksheet.columns
    ):

        column_index = (
            column_cells[0].column
        )

        maximum_length = 0

        for cell in column_cells:

            if cell.value is not None:

                maximum_length = max(
                    maximum_length,
                    len(
                        str(
                            cell.value
                        )
                    ),
                )

        adjusted_width = min(
            maximum_length + 2,
            55,
        )

        worksheet.column_dimensions[
            get_column_letter(
                column_index
            )
        ].width = adjusted_width


workbook.save(
    excel_output_path
)


# ============================================================
# 28. Final console output
# ============================================================

print("\n" + "=" * 70)
print("ABLATION STUDY COMPLETED")
print("=" * 70)

print(
    "\nRepresentation-component results "
    "at the top-2% budget:"
)

print(
    representation_top2_df.round(
        3
    ).to_string(
        index=False
    )
)

print(
    "\nSpatial/context/reliability results "
    "at the top-2% budget:"
)

print(
    module_top2_df.round(
        3
    ).to_string(
        index=False
    )
)

print(
    "\nExcel workbook saved to:\n"
    f"{excel_output_path.resolve()}"
)

print(
    "\nAll outputs saved in:\n"
    f"{OUTPUT_DIR.resolve()}"
)

print(
    "\nUpdated paper-oriented figures:"
)

print(
    "1. Representation_Ablation_Top2_Heatmap.png"
)

print(
    "2. Module_Ablation_Top2_Heatmap.png"
)

print(
    "3. Representation_Core_Budget_Sensitivity.png"
)

print(
    "4. Representation_Removal_Budget_Sensitivity.png"
)

print(
    "5. Full_Module_Budget_Sensitivity.png"
)

print(
    "6. Pure_Module_Budget_Sensitivity.png"
)

print(
    "\nAblation scope record:"
)

print(
    OUTPUT_DIR
    / "Ablation_Study_Scope.txt"
)